# Chapter 4. Spatial Descriptive Statistics and Hotspot Analysis

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Install the packages

Colab does not ship the geopandas stack, so install it once. Skip this cell in a local Anaconda environment.

In [ ]:
!pip install -q geopandas libpysal esda statsmodels mapclassify

## Step 1. Load the data and run three checks

144 synthetic communes. Confirm three things before anything else: at least 30 units, a projected coordinate system, and no missing values.

In [ ]:
import geopandas as gpd

PATH = "data/phnompenh_communes.geojson"
COL = "informal_rate"

gdf = gpd.read_file(PATH).dropna(subset=[COL])
print("units:", len(gdf), "| at least 30?", len(gdf) >= 30)
print("CRS:", gdf.crs, "| projected?", not gdf.crs.is_geographic)
gdf.head()

## Step 2. The spatial weights matrix: who counts as a neighbour

Queen contiguity is the default. **The choice changes the result**, so run at least two definitions and confirm your finding holds.

In [ ]:
from libpysal import weights

w = weights.Queen.from_dataframe(gdf, use_index=False)
w.transform = "r"
print("islands (no neighbours):", w.islands)
print("mean neighbours:", round(sum(len(v) for v in w.neighbors.values()) / w.n, 2))

## Step 3. Global Moran's I: is there clustering overall

One number for the whole map. It tells you clustering exists; it does not tell you where.

In [ ]:
from esda.moran import Moran

mi = Moran(gdf[COL], w, permutations=999)
print(f"Global Moran's I = {mi.I:.3f}   pseudo p = {mi.p_sim:.3f}")

## Step 4. LISA: where, and of what kind

High-High is a hotspot, Low-Low a coldspot, High-Low and Low-High are spatial outliers. A multiple comparisons correction is applied because a test is repeated at every unit.

In [ ]:
import numpy as np
from esda.moran import Moran_Local
from statsmodels.stats.multitest import multipletests

lm = Moran_Local(gdf[COL], w, permutations=999)
sig = multipletests(lm.p_sim, alpha=0.05, method="fdr_bh")[0]
labels = np.array(["ns"] * len(gdf), dtype=object)
names = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
for q, nm in names.items():
    labels[(lm.q == q) & sig] = nm
gdf["lisa_label"] = labels
print(gdf["lisa_label"].value_counts().to_string())

## Step 5. Getis-Ord Gi*: drawing hotspot boundaries

Where LISA classifies into types, Gi* returns a continuous z-score, which is what you want when the output has to become a boundary.

In [ ]:
from esda.getisord import G_Local

gi = G_Local(gdf[COL], w, star=True, permutations=999)
gdf["gi_z"] = gi.Zs
print("hotspots (z > 1.96):", int((gdf.gi_z > 1.96).sum()))
print("coldspots (z < -1.96):", int((gdf.gi_z < -1.96).sum()))

## Step 6. Draw the maps

Three panels: raw values, LISA cluster types, and the Gi* surface. Reproduces Figure 4-1.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
gdf.plot(column=COL, cmap="OrRd", legend=True, ax=ax[0], edgecolor="white", linewidth=0.3)
ax[0].set_title("(a) Informal housing rate")

colours = {"High-High": "#d7191c", "Low-Low": "#2c7bb6",
           "High-Low": "#fdae61", "Low-High": "#abd9e9", "ns": "#e8e8e8"}
for k, c in colours.items():
    sub = gdf[gdf.lisa_label == k]
    if len(sub):
        sub.plot(ax=ax[1], color=c, edgecolor="white", linewidth=0.3, label=f"{k} (n={len(sub)})")
ax[1].legend(fontsize=8)
ax[1].set_title("(b) LISA cluster types")

gdf.plot(column="gi_z", cmap="RdBu_r", legend=True, ax=ax[2],
         edgecolor="white", linewidth=0.3, vmin=-3, vmax=3)
ax[2].set_title("(c) Getis-Ord Gi*")
for a in ax:
    a.set_axis_off()
plt.tight_layout()
plt.show()

## Exporting the result

To finish the maps in QGIS, download the file below and open it there.

In [ ]:
gdf.to_file("results.geojson", driver="GeoJSON")
print("saved -> results.geojson")

---

**What to do next.** Compare against Section 4.4 of the book. If a number differs, check the weights definition first. The land surface temperature produced in Chapter 7 can be fed into this notebook as an alternative input.